# Design and Evaluation of a Transformer-Based Early Warning System for At-Risk Student Identification in Higher Education

**Student:** Eashkumar Kaki | **Student No:** 35057938 | **Supervisor:** Tom Akinbileje

**Dataset:** OULAD (Open University Learning Analytics Dataset)

**Research Question:** To what extent can transformer-based sequence models, applied to LMS interaction logs, assessment history and learner demographics from OULAD, outperform traditional machine learning baselines in early student performance prediction while maintaining interpretability, fairness and suitability for tutor-led academic intervention?

---

**Pipeline:**
1. Load and explore OULAD data
2. Clean data and define target variable
3. Feature engineering (early-semester engagement + assessment features)
4. Merge into final modelling dataframe
5. Encode categorical variables
6. Handle class imbalance (SMOTE)
7. Baseline models: Logistic Regression, Random Forest, XGBoost
8. Sequence models: LSTM, Transformer
9. Explainability: SHAP and attention analysis
10. Fairness evaluation across demographic subgroups
11. Model comparison and evaluation plots
12. Dashboard prototype (tutor-facing risk view)


## 1. Setup and Imports

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Modelling
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (f1_score, roc_auc_score, average_precision_score,
                              classification_report, confusion_matrix, roc_curve,
                              precision_recall_curve, ConfusionMatrixDisplay)
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE

# Deep learning
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (LSTM, Dense, Dropout, Input, MultiHeadAttention,
                                       LayerNormalization, GlobalAveragePooling1D)
from tensorflow.keras.callbacks import EarlyStopping

# Explainability
import shap

# Plot styling
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


## 2. Load OULAD Data

The OULAD files need to be located under `/kaggle/input` first, since Kaggle sometimes nests the dataset folder differently (for example under `/kaggle/input/datasets/anlgrbz/...` instead of directly under `/kaggle/input/...`). The cell below searches for `studentInfo.csv` and sets `DATA_DIR` to whichever folder it is actually found in, so the path does not need to be hardcoded.

In [ ]:
# Locate the folder containing the OULAD CSV files under /kaggle/input,
# regardless of how deeply Kaggle has nested it.
import os

DATA_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'studentInfo.csv' in files:
        DATA_DIR = root + '/'
        break

if DATA_DIR is None:
    print("studentInfo.csv not found under /kaggle/input. Files found instead:")
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            print(os.path.join(root, f))
else:
    print("Dataset folder found:", DATA_DIR)


In [ ]:
studentInfo = pd.read_csv(DATA_DIR + "studentInfo.csv")
studentVle = pd.read_csv(DATA_DIR + "studentVle.csv")
studentAssessment = pd.read_csv(DATA_DIR + "studentAssessment.csv")
assessments = pd.read_csv(DATA_DIR + "assessments.csv")

print("studentInfo:", studentInfo.shape)
print("studentVle:", studentVle.shape)
print("studentAssessment:", studentAssessment.shape)
print("assessments:", assessments.shape)

studentInfo.head()


## 3. Exploratory Data Analysis

In [ ]:
print("Final result distribution:")
print(studentInfo['final_result'].value_counts())
print("\nPercentage:")
print(studentInfo['final_result'].value_counts(normalize=True) * 100)

fig, ax = plt.subplots(figsize=(8, 5))
studentInfo['final_result'].value_counts().plot(kind='bar', color='#4A90D9', ax=ax)
ax.set_title('Distribution of Final Result (OULAD)', fontsize=13, fontweight='bold')
ax.set_xlabel('Final Result')
ax.set_ylabel('Number of Students')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('final_result_distribution.png', dpi=150)
plt.show()


In [ ]:
print("Missing values per column:")
print(studentInfo.isnull().sum()[studentInfo.isnull().sum() > 0])


## 4. Data Cleaning and Target Variable

The binary target `at_risk` is defined as:
- **1 (At-Risk)** = Fail or Withdrawn
- **0 (Not At-Risk)** = Pass or Distinction

This binary framing matches the early-warning purpose of the system. The priority is catching students who need intervention, not fine-grained grade prediction.

In [ ]:
# Fill missing IMD band with explicit 'Unknown' category rather than dropping
studentInfo['imd_band'] = studentInfo['imd_band'].fillna('Unknown')

# Remove duplicate student-module-presentation records if any
studentInfo = studentInfo.drop_duplicates(subset=['id_student', 'code_module', 'code_presentation'])

# Binary target
studentInfo['at_risk'] = studentInfo['final_result'].apply(
    lambda x: 1 if x in ['Fail', 'Withdrawn'] else 0
)

print("Class balance:")
print(studentInfo['at_risk'].value_counts())
print(studentInfo['at_risk'].value_counts(normalize=True).round(3))


## 5. Feature Engineering

**Early-window strategy:** only data from the first 4 weeks (weeks 0-4) of each module presentation is used, since this is the point at which meaningful intervention is still possible for a genuine early-warning system.

Three feature groups are built:
1. **Weekly VLE click sequence** (for LSTM/Transformer): clicks per week, weeks 0-4
2. **VLE summary statistics** (for baseline models): total clicks, active days, average daily clicks
3. **Early assessment performance**: average score and number of assessments submitted in the early window

In [ ]:
# 5a. Weekly VLE clicks: convert date (days since start) into week number
studentVle['week'] = studentVle['date'] // 7
early_vle = studentVle[(studentVle['week'] >= 0) & (studentVle['week'] <= 4)]

weekly_clicks = early_vle.groupby(
    ['id_student', 'code_module', 'code_presentation', 'week']
)['sum_click'].sum().reset_index()

# Pivot to wide format, one column per week (needed for sequence models)
weekly_pivot = weekly_clicks.pivot_table(
    index=['id_student', 'code_module', 'code_presentation'],
    columns='week', values='sum_click', fill_value=0
).reset_index()

weekly_pivot.columns = (
    ['id_student', 'code_module', 'code_presentation'] +
    [f'week_{int(c)}_clicks' for c in weekly_pivot.columns[3:]]
)

print("Weekly pivot shape:", weekly_pivot.shape)
weekly_pivot.head()


In [ ]:
# 5b. VLE summary statistics (for baseline tree/linear models)
vle_summary = early_vle.groupby(
    ['id_student', 'code_module', 'code_presentation']
).agg(
    total_clicks=('sum_click', 'sum'),
    active_days=('date', 'nunique'),
    avg_clicks_per_day=('sum_click', 'mean')
).reset_index()

vle_summary.head()


In [ ]:
# 5c. Early assessment performance
assess_merged = studentAssessment.merge(assessments, on='id_assessment', how='left')
early_assess = assess_merged[assess_merged['date'] <= 28]  # weeks 0-4

assess_summary = early_assess.groupby('id_student').agg(
    early_avg_score=('score', 'mean'),
    early_num_submitted=('id_assessment', 'count')
).reset_index()

assess_summary.head()


## 6. Merge Into Final Modelling Dataframe

In [ ]:
df = studentInfo.merge(vle_summary, on=['id_student', 'code_module', 'code_presentation'], how='left')
df = df.merge(weekly_pivot, on=['id_student', 'code_module', 'code_presentation'], how='left')
df = df.merge(assess_summary, on='id_student', how='left')

# Missing rows mean no early activity/assessment record was found, i.e. zero engagement
fill_cols = [c for c in df.columns if 'clicks' in c or 'score' in c or 'submitted' in c or 'active_days' in c]
df[fill_cols] = df[fill_cols].fillna(0)

print("Final merged dataframe shape:", df.shape)
print("\nRemaining nulls:")
print(df.isnull().sum()[df.isnull().sum() > 0])

df.head()


## 7. Encode Categorical Variables

In [ ]:
cat_cols = ['gender', 'region', 'highest_education', 'imd_band', 'age_band', 'disability']
encoders = {}

for c in cat_cols:
    le = LabelEncoder()
    df[c + '_enc'] = le.fit_transform(df[c].astype(str))
    encoders[c] = le

print("Encoded categorical columns:", [c + '_enc' for c in cat_cols])


## 8. Build Feature Matrix

In [ ]:
week_cols = sorted(
    [c for c in df.columns if c.startswith('week_') and c.endswith('_clicks')],
    key=lambda x: int(x.split('_')[1])
)

feature_cols = (
    [c + '_enc' for c in cat_cols] +
    ['num_of_prev_attempts', 'studied_credits'] +
    ['total_clicks', 'active_days', 'avg_clicks_per_day'] +
    week_cols +
    ['early_avg_score', 'early_num_submitted']
)

X = df[feature_cols]
y = df['at_risk']

print("Feature matrix shape:", X.shape)
print("Features used:", feature_cols)


## 9. Train/Test Split and Class Imbalance Handling

**SMOTE** is applied to the training set only (never test data, to avoid leakage) for the baseline models.

For the sequence models (LSTM/Transformer), **class weighting** is used instead during training. This is more standard for sequential data and avoids synthetically distorting temporal patterns.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

sm = SMOTE(random_state=RANDOM_STATE)
X_train_bal, y_train_bal = sm.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", pd.Series(y_train_bal).value_counts().to_dict())


## 10. Baseline Models: Logistic Regression, Random Forest, XGBoost

In [ ]:
results = {}

def evaluate(name, y_true, y_pred, y_proba):
    results[name] = {
        'F1': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_proba),
        'PR-AUC': average_precision_score(y_true, y_proba)
    }
    print(f"\n{name}")
    print(classification_report(y_true, y_pred, target_names=['Not At-Risk', 'At-Risk']))
    return results[name]


In [ ]:
# --- Logistic Regression ---
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_bal, y_train_bal)
pred_lr = lr.predict(X_test)
proba_lr = lr.predict_proba(X_test)[:, 1]
evaluate('Logistic Regression', y_test, pred_lr, proba_lr)


In [ ]:
# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=300, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_bal, y_train_bal)
pred_rf = rf.predict(X_test)
proba_rf = rf.predict_proba(X_test)[:, 1]
evaluate('Random Forest', y_test, pred_rf, proba_rf)


In [ ]:
# --- XGBoost ---
xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
)
xgb.fit(X_train_bal, y_train_bal)
pred_xgb = xgb.predict(X_test)
proba_xgb = xgb.predict_proba(X_test)[:, 1]
evaluate('XGBoost', y_test, pred_xgb, proba_xgb)


## 11. Sequence Models: LSTM and Transformer

The weekly click sequence (5 timesteps: weeks 0-4) is reshaped into `(n_students, n_weeks, 1)` format for the recurrent and attention-based models.

In [ ]:
seq_data = df[week_cols].values.astype('float32')
seq_data = seq_data.reshape(seq_data.shape[0], seq_data.shape[1], 1)

# Normalise (avoid divide-by-zero if all clicks are 0)
seq_max = seq_data.max() if seq_data.max() > 0 else 1.0
seq_data_norm = seq_data / seq_max

X_seq_train, X_seq_test, y_seq_train, y_seq_test = train_test_split(
    seq_data_norm, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Sequence shape:", seq_data_norm.shape)
print("Train:", X_seq_train.shape, "Test:", X_seq_test.shape)

# Class weights for imbalance (used instead of SMOTE for sequence data)
class_weights = compute_class_weight('balanced', classes=np.unique(y_seq_train), y=y_seq_train)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}
print("Class weights:", class_weight_dict)


In [ ]:
# --- LSTM ---
lstm_model = Sequential([
    Input(shape=(X_seq_train.shape[1], 1)),
    LSTM(64, return_sequences=True),
    Dropout(0.3),
    LSTM(32, return_sequences=False),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])
lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm_model.summary()


In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_lstm = lstm_model.fit(
    X_seq_train, y_seq_train,
    validation_split=0.15,
    epochs=40,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
lstm_proba = lstm_model.predict(X_seq_test).flatten()
lstm_pred = (lstm_proba > 0.5).astype(int)
evaluate('LSTM', y_seq_test, lstm_pred, lstm_proba)


In [ ]:
# Plot LSTM training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(history_lstm.history['loss'], label='Train Loss', color='#1E2761')
axes[0].plot(history_lstm.history['val_loss'], label='Validation Loss', color='#E67E22')
axes[0].set_title('LSTM: Loss Curve'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(history_lstm.history['accuracy'], label='Train Accuracy', color='#1E2761')
axes[1].plot(history_lstm.history['val_accuracy'], label='Validation Accuracy', color='#E67E22')
axes[1].set_title('LSTM: Accuracy Curve'); axes[1].set_xlabel('Epoch'); axes[1].legend()

plt.tight_layout()
plt.savefig('lstm_training_curves.png', dpi=150)
plt.show()


### Transformer Model

A lightweight encoder-only transformer using multi-head self-attention over the weekly click sequence. Attention weights can be inspected afterwards to see which weeks the model finds most informative, forming a second explainability layer alongside SHAP.

In [ ]:
def build_transformer(seq_len, d_model=32, num_heads=2, ff_dim=32, dropout=0.2):
    # Two outputs are produced from one forward pass: the risk prediction
    # and the raw attention scores. Re-calling layers after training to
    # pull out attention creates a disconnected graph in Keras, so both
    # outputs are defined together here instead.
    inputs = Input(shape=(seq_len, 1))
    x = Dense(d_model)(inputs)  # project single click value into d_model dimensions

    mha = MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
    attn_output, attn_scores = mha(x, x, return_attention_scores=True)
    x = LayerNormalization(epsilon=1e-6)(x + attn_output)

    ff = Dense(ff_dim, activation='relu')(x)
    ff = Dense(d_model)(ff)
    x = LayerNormalization(epsilon=1e-6)(x + ff)

    x = GlobalAveragePooling1D()(x)
    x = Dropout(dropout)(x)
    x = Dense(16, activation='relu')(x)
    prediction = Dense(1, activation='sigmoid', name='risk_prediction')(x)

    # Training model: single output, used for fit/predict/evaluate as normal
    train_model = Model(inputs, prediction, name='transformer_early_warning')

    # Attention model: shares the same layer weights, exposes attention scores
    attn_model = Model(inputs, attn_scores, name='transformer_attention_extractor')

    return train_model, attn_model

transformer_model, transformer_attn_model = build_transformer(X_seq_train.shape[1])
transformer_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
transformer_model.summary()


In [ ]:
history_tr = transformer_model.fit(
    X_seq_train, y_seq_train,
    validation_split=0.15,
    epochs=40,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
tr_proba = transformer_model.predict(X_seq_test).flatten()
tr_pred = (tr_proba > 0.5).astype(int)
evaluate('Transformer', y_seq_test, tr_pred, tr_proba)


In [ ]:
# Plot Transformer training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(history_tr.history['loss'], label='Train Loss', color='#1E2761')
axes[0].plot(history_tr.history['val_loss'], label='Validation Loss', color='#E67E22')
axes[0].set_title('Transformer: Loss Curve'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(history_tr.history['accuracy'], label='Train Accuracy', color='#1E2761')
axes[1].plot(history_tr.history['val_accuracy'], label='Validation Accuracy', color='#E67E22')
axes[1].set_title('Transformer: Accuracy Curve'); axes[1].set_xlabel('Epoch'); axes[1].legend()

plt.tight_layout()
plt.savefig('transformer_training_curves.png', dpi=150)
plt.show()


## 12. Model Comparison

In [ ]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('PR-AUC', ascending=False)
print(results_df.round(3))

fig, ax = plt.subplots(figsize=(10, 5))
results_df.plot(kind='bar', ax=ax, color=['#4A90D9', '#1E2761', '#E67E22'])
ax.set_title('Model Comparison: F1 / ROC-AUC / PR-AUC', fontsize=13, fontweight='bold')
ax.set_ylabel('Score')
ax.set_xlabel('Model')
plt.xticks(rotation=20)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()


In [ ]:
# ROC curves for all models
fig, ax = plt.subplots(figsize=(7, 6))
for name, proba in [('Logistic Regression', proba_lr), ('Random Forest', proba_rf),
                     ('XGBoost', proba_xgb), ('LSTM', lstm_proba), ('Transformer', tr_proba)]:
    yt = y_test if name in ['Logistic Regression', 'Random Forest', 'XGBoost'] else y_seq_test
    fpr, tpr, _ = roc_curve(yt, proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(yt, proba):.3f})")

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_title('ROC Curves: All Models', fontsize=13, fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('roc_curves_all_models.png', dpi=150)
plt.show()


In [ ]:
# Confusion matrix for best model (adjust variable name if a different model wins)
best_model_name = results_df.index[0]
print("Best model by PR-AUC:", best_model_name)

pred_map = {
    'Logistic Regression': pred_lr, 'Random Forest': pred_rf, 'XGBoost': pred_xgb,
    'LSTM': lstm_pred, 'Transformer': tr_pred
}
truth_map = {
    'Logistic Regression': y_test, 'Random Forest': y_test, 'XGBoost': y_test,
    'LSTM': y_seq_test, 'Transformer': y_seq_test
}

cm = confusion_matrix(truth_map[best_model_name], pred_map[best_model_name])
disp = ConfusionMatrixDisplay(cm, display_labels=['Not At-Risk', 'At-Risk'])
disp.plot(cmap='Blues')
plt.title(f'Confusion Matrix: {best_model_name}')
plt.savefig('confusion_matrix_best_model.png', dpi=150)
plt.show()


## 13. Explainability: SHAP (Baseline Models)

SHAP values show which features drive each prediction, giving tutors an interpretable reason for a risk flag rather than a black-box score.

In [ ]:
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test, feature_names=feature_cols, show=False)
plt.title('SHAP Summary: Feature Impact on At-Risk Prediction (XGBoost)')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Global feature importance (mean absolute SHAP value)
mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({'feature': feature_cols, 'mean_abs_shap': mean_abs_shap})
importance_df = importance_df.sort_values('mean_abs_shap', ascending=False)
print(importance_df.head(10))


### Transformer Attention Analysis

Attention weights are extracted from the transformer's `MultiHeadAttention` layer to see which weeks the model focuses on when making a prediction, giving a second, complementary form of explainability alongside SHAP.

In [ ]:
# transformer_attn_model was built alongside transformer_model and
# shares the same trained weights, so it can be called directly with
# no need to re-wire the graph after training.
sample_attn = transformer_attn_model.predict(X_seq_test[:20])
avg_attn = sample_attn.mean(axis=(0, 1))  # average over samples and heads

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(avg_attn, cmap='Blues', annot=True, fmt='.2f',
            xticklabels=[f'W{i}' for i in range(avg_attn.shape[1])],
            yticklabels=[f'W{i}' for i in range(avg_attn.shape[0])], ax=ax)
ax.set_title('Transformer: Average Attention Weights Across Weeks')
plt.tight_layout()
plt.savefig('transformer_attention_heatmap.png', dpi=150)
plt.show()


## 14. Fairness Evaluation Across Demographic Subgroups

This section checks whether the best-performing model's predictive performance is consistent across gender, disability status and IMD band. This is a requirement given the ethics approval for this project and the responsible-AI framing of the research question.

In [ ]:
def fairness_report(feature_name, y_true, y_pred, y_proba, group_labels):
    report_df = pd.DataFrame({
        'group': group_labels.values,
        'y_true': y_true.values if hasattr(y_true, 'values') else y_true,
        'y_pred': y_pred,
        'y_proba': y_proba
    })

    rows = []
    for g, sub in report_df.groupby('group'):
        if sub['y_true'].nunique() < 2:
            continue
        rows.append({
            'group': g,
            'n': len(sub),
            'F1': f1_score(sub['y_true'], sub['y_pred']),
            'ROC-AUC': roc_auc_score(sub['y_true'], sub['y_proba']),
            'Positive_rate_predicted': sub['y_pred'].mean()
        })
    return pd.DataFrame(rows)


# Use XGBoost predictions (or swap to best_model_name equivalent) against test indices
gender_groups = df.loc[X_test.index, 'gender']
disability_groups = df.loc[X_test.index, 'disability']
imd_groups = df.loc[X_test.index, 'imd_band']

print("Fairness by Gender:")
print(fairness_report('gender', y_test, pred_xgb, proba_xgb, gender_groups))

print("\nFairness by Disability Status:")
print(fairness_report('disability', y_test, pred_xgb, proba_xgb, disability_groups))

print("\nFairness by IMD Band:")
print(fairness_report('imd_band', y_test, pred_xgb, proba_xgb, imd_groups))


In [ ]:
# Fairness comparison: F1 by gender and disability
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

gender_fair = fairness_report('gender', y_test, pred_xgb, proba_xgb, gender_groups)
axes[0].bar(gender_fair['group'], gender_fair['F1'], color='#4A90D9')
axes[0].set_title('F1 Score by Gender'); axes[0].set_ylim(0, 1)

disability_fair = fairness_report('disability', y_test, pred_xgb, proba_xgb, disability_groups)
axes[1].bar(disability_fair['group'], disability_fair['F1'], color='#E67E22')
axes[1].set_title('F1 Score by Disability Status'); axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('fairness_subgroup_comparison.png', dpi=150)
plt.show()


## 15. Dashboard Prototype: Tutor-Facing Risk View

A simplified, notebook-based prototype of the intervention dashboard: for a sample of students, it shows their predicted risk level, confidence score, top contributing factors (from SHAP), and a recommended tutor action. This is the artefact used during the UAT session with academic stakeholders.

In [ ]:
def generate_risk_dashboard(student_indices, X_data, model, explainer, feature_names, original_df):
    dashboard_rows = []
    proba = model.predict_proba(X_data.loc[student_indices])[:, 1]
    shap_vals = explainer.shap_values(X_data.loc[student_indices])

    for i, idx in enumerate(student_indices):
        risk_score = proba[i]
        risk_level = 'High' if risk_score >= 0.66 else 'Medium' if risk_score >= 0.33 else 'Low'

        # Top 3 contributing features for this student
        contrib = pd.Series(shap_vals[i], index=feature_names).abs().sort_values(ascending=False)
        top_factors = contrib.head(3).index.tolist()

        action = (
            "Immediate tutor outreach recommended" if risk_level == 'High' else
            "Monitor engagement over next 2 weeks" if risk_level == 'Medium' else
            "No action needed"
        )

        dashboard_rows.append({
            'student_id': original_df.loc[idx, 'id_student'],
            'risk_score': round(risk_score, 3),
            'risk_level': risk_level,
            'top_contributing_factors': ", ".join(top_factors),
            'recommended_action': action
        })

    return pd.DataFrame(dashboard_rows)


sample_students = X_test.index[:15]
dashboard = generate_risk_dashboard(sample_students, X_test, xgb, explainer, feature_cols, df)
dashboard = dashboard.sort_values('risk_score', ascending=False)
dashboard


In [ ]:
# Visual dashboard-style risk view
fig, ax = plt.subplots(figsize=(10, 6))
colors = dashboard['risk_level'].map({'High': '#E74C3C', 'Medium': '#E67E22', 'Low': '#2ECC71'})
ax.barh(dashboard['student_id'].astype(str), dashboard['risk_score'], color=colors)
ax.set_xlabel('Predicted Risk Score')
ax.set_ylabel('Student ID')
ax.set_title('Early Warning Dashboard: Student Risk Overview', fontsize=13, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('dashboard_risk_overview.png', dpi=150)
plt.show()


## 16. Save Models and Outputs

Save trained models and the results table for use in the report and PPT.

In [ ]:
import joblib

# Save baseline models
joblib.dump(lr, 'logistic_regression_model.pkl')
joblib.dump(rf, 'random_forest_model.pkl')
joblib.dump(xgb, 'xgboost_model.pkl')

# Save deep learning models
lstm_model.save('lstm_model.keras')
transformer_model.save('transformer_model.keras')

# Save results table
results_df.to_csv('model_comparison_results.csv')

# Save dashboard sample
dashboard.to_csv('dashboard_sample_output.csv', index=False)

print("All models and outputs saved.")
print("\nFinal Results Summary:")
print(results_df.round(3))


## 17. Zip All Outputs for Download

In [ ]:
import shutil
import os

# Collect every output file this notebook has generated into one folder, then zip it.
output_files = [
    'final_result_distribution.png',
    'lstm_training_curves.png',
    'transformer_training_curves.png',
    'model_comparison.png',
    'roc_curves_all_models.png',
    'confusion_matrix_best_model.png',
    'shap_summary.png',
    'transformer_attention_heatmap.png',
    'fairness_subgroup_comparison.png',
    'dashboard_risk_overview.png',
    'model_comparison_results.csv',
    'dashboard_sample_output.csv',
    'logistic_regression_model.pkl',
    'random_forest_model.pkl',
    'xgboost_model.pkl',
    'lstm_model.keras',
    'transformer_model.keras',
]

bundle_dir = 'project_outputs'
os.makedirs(bundle_dir, exist_ok=True)

copied, missing = [], []
for f in output_files:
    if os.path.exists(f):
        shutil.copy(f, os.path.join(bundle_dir, f))
        copied.append(f)
    else:
        missing.append(f)

zip_path = shutil.make_archive('OULAD_Early_Warning_System_Outputs', 'zip', bundle_dir)

print(f"Zipped {len(copied)} files into: {zip_path}")
if missing:
    print("\nNot found (check earlier cells ran successfully):")
    for m in missing:
        print(" -", m)

print("\nOn Kaggle: open the notebook's Output tab (right-hand panel) after committing/running,")
print("and download 'OULAD_Early_Warning_System_Outputs.zip', one file with everything included.")


---
### Notebook complete.

**Everything is bundled into one download:** `OULAD_Early_Warning_System_Outputs.zip`

Contains:
- `final_result_distribution.png`
- `lstm_training_curves.png`
- `transformer_training_curves.png`
- `model_comparison.png`
- `roc_curves_all_models.png`
- `confusion_matrix_best_model.png`
- `shap_summary.png`
- `transformer_attention_heatmap.png`
- `fairness_subgroup_comparison.png`
- `dashboard_risk_overview.png`
- `model_comparison_results.csv`
- `dashboard_sample_output.csv`
- Saved model files (`.pkl` and `.keras`)

On Kaggle, after the notebook finishes running (or after clicking Save Version, Run All), the Output tab on the right-hand panel has the zip ready to download in one click instead of grabbing each file individually.

Next step: export the dashboard sample and UAT results into the Google Form summary for the Evaluation section of the report.
